# Parameter identification with data fitting
After conducting the experiment, we compare the recorded data with the calculated behavior of the mathematical model of the DC motor.

The TRIK program `Data_collection.qrs` sweeps motor power from −100 % to +100 % in steps of 5 %. For each power level it runs the motors for ~2 s and logs one line every ~110 ms to `trik_data_bat.txt`:

| column | meaning | unit |
|---|---|---|
| 0 | motor power setpoint | % |
| 1 | left wheel encoder (E4) | deg |
| 2 | right wheel encoder (E3) | deg |
| 3 | time since start of this power level | ms |
| 4 | battery voltage | V |

Because the battery voltage is logged, we can use the **actual voltage** applied to the motor, $U = \frac{power}{100}\,U_{bat}$, instead of a guessed constant.

In [ ]:
from google.colab import files    # file work module
uploaded = files.upload()         # upload file context window

Using the *loadtxt* function from the *numpy* module we read all five columns. The file holds many experiments one after another, so we split it into separate runs wherever the power setpoint changes, and convert the readings to SI units.

In [ ]:
import numpy as np                  # array work module
import matplotlib.pyplot as plt     # data visualization
from scipy import optimize          # data fitting
from scipy.integrate import odeint  # ordinary differential equation solver

DATA_FILE = 'trik_data_bat.txt'     # recorded experiment
data = np.loadtxt(DATA_FILE)        # columns: power, encL, encR, time_ms, battery
power_all, encL_all, encR_all, time_all, bat_all = data.T

starts = np.r_[0, np.flatnonzero(np.diff(power_all) != 0) + 1, len(power_all)]  # run boundaries
runs = []
for a, b in zip(starts[:-1], starts[1:]):
    runs.append({
        'power': power_all[a],                                  # setpoint, %
        'time':  (time_all[a:b] - time_all[a]) / 1000,          # SI conversion, s
        'L':     np.deg2rad(encL_all[a:b]),                     # SI conversion, rad
        'R':     np.deg2rad(encR_all[a:b]),                     # SI conversion, rad
        'U':     power_all[a] / 100 * bat_all[a:b].mean(),      # applied voltage, V
    })
print(f'{len(runs)} runs, power from {runs[0]["power"]:.0f} to {runs[-1]["power"]:.0f} %')
print(f'battery voltage: {bat_all.min():.2f} .. {bat_all.max():.2f} V')

Choose which experiment to analyse in detail. Full power gives the cleanest response, like the single experiment in the original notebook.

In [ ]:
POWER = 100                         # power level of the run to analyse, %
WHEEL = 'L'                         # 'L' or 'R'

run = next(r for r in runs if r['power'] == POWER)
time, angle, u = run['time'], run[WHEEL], run['U']
print(f'power {POWER} %, wheel {WHEEL}, U = {u:.2f} V, {len(time)} samples')
plt.plot(time, angle, 'r.-')        # graph plot
plt.xlabel('t, s'); plt.ylabel('angle, rad'); plt.grid()

Next, we define the motor rotation angle function to find optimal values of the steady-state speed $\omega_{ss}$ and $T_m$:
$$\theta(t) = \omega_{ss}\left(t - T_m + T_m e^{-t/T_m}\right), \qquad \omega_{ss} = \frac{U}{k_e}$$
Having real measurements of rotation angle versus time, we use *optimize.curve_fit*. The initial guess for the speed is taken from the data itself, so it works for any power level and direction.

In [ ]:
def model(t, speed, T_m):
    return speed * t - speed * T_m + speed * T_m * np.exp(-t / T_m)  # definition of the angle equation

def fit_run(t, ang):
    p0 = [ang[-1] / t[-1], 0.2]                                      # initial guess from the data
    popt, _ = optimize.curve_fit(model, t, ang, p0=p0,
                                 bounds=([-np.inf, 1e-3], [np.inf, 5]))  # T_m must stay positive
    return popt

popt = fit_run(time, angle)                                          # data fitting
speed, T_m = popt
k_e = u / speed                                                      # back EMF constant from U = k_e * w_ss
print(f'w_ss = {speed:.3f} rad/s')
print(f'T_m  = {T_m:.3f} s')
print(f'k_e  = {k_e:.3f} V*s/rad')

For clarity, let's compare the graphs of the equation with the found coefficients and experimental measurements. The experimental graph usually accelerates slightly slower at the very beginning, since some physical processes were not accounted for in the model: friction, electromagnetic dynamics and the delay of the TRIK motor driver.

In [ ]:
equation = model(time, *popt)                                        # numerical modeling
plt.figure(figsize=(10, 6), dpi=100)                                 # plot window parameters
plt.plot(time, equation, 'b--', linewidth=3, label='equation')       # numerical modelling plot
plt.plot(time, angle, 'r.-', linewidth=1, label='experiment')        # experimental data plot
plt.xlabel('t, s'); plt.ylabel('angle, rad'); plt.grid(); plt.legend()

## Identification over the whole power sweep
The file contains a run for every power level, so we can fit every run for both wheels. Runs where the wheel barely moved (e.g. 0 %) are skipped.

In [ ]:
rows = []
for r in runs:
    for wheel in ('L', 'R'):
        ang = r[wheel]
        if abs(ang[-1]) < 1:                                         # wheel did not move (dead zone)
            continue
        try:
            w, tm = fit_run(r['time'], ang)
        except RuntimeError:                                         # fit did not converge
            continue
        rows.append((r['power'], r['U'], wheel, w, tm))

res_power = np.array([x[0] for x in rows])
res_U     = np.array([x[1] for x in rows])
res_wheel = np.array([x[2] for x in rows])
res_w     = np.array([x[3] for x in rows])
res_Tm    = np.array([x[4] for x in rows])

print(' power     U    wheel   w_ss     T_m    k_e=U/w')
for p, U, wh, w, tm in rows:
    print(f'{p:6.0f} {U:7.2f}    {wh}   {w:7.2f}  {tm:6.3f}  {U / w:6.3f}')

The ideal model says $\omega_{ss} = U / k_e$, i.e. a straight line through the origin. Plotting the fitted steady-state speed against the applied voltage shows how well the TRIK motor follows this: the curve is not a single line through the origin: the speed jumps to ~4 rad/s already at 10 % power and then grows roughly linearly with a smaller slope (the TRIK motor driver is not a linear voltage source). That is why $k_e$ computed at a single power level is only valid near that operating point.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(14, 5), dpi=100)
for wheel, c in (('L', 'tab:blue'), ('R', 'tab:orange')):
    m = res_wheel == wheel
    ax[0].plot(res_U[m], res_w[m], 'o', color=c, label=f'wheel {wheel}')
    ax[1].plot(res_U[m], res_Tm[m], 'o', color=c, label=f'wheel {wheel}')

hi = np.abs(res_power) >= 50                                         # upper half of the range
k_lin = np.sum(res_U[hi] * res_w[hi]) / np.sum(res_U[hi] ** 2)       # least squares through origin: w = k_lin * U
U_line = np.linspace(res_U.min(), res_U.max(), 100)
ax[0].plot(U_line, k_lin * U_line, 'k--', label=f'w = U / {1 / k_lin:.3f} (|power| >= 50 %)')
ax[0].set_xlabel('U, V'); ax[0].set_ylabel('w_ss, rad/s'); ax[0].grid(); ax[0].legend()
ax[1].set_xlabel('U, V'); ax[1].set_ylabel('T_m, s'); ax[1].grid(); ax[1].legend()

print(f'k_e (fit over |power| >= 50 %) = {1 / k_lin:.3f} V*s/rad')
print(f'T_m (median over |power| >= 50 %) = {np.median(res_Tm[hi]):.3f} s')

## Comparison with last year's worked example
Last year's group (`05_Motor_parameter_identification/comments.md`, scripts `plot.py` and `plot_omega.py`) did not fit the exponential model. Instead, for each power level they:
1. took $\omega$ as the slope of the **last 60 %** of the position samples (the linear, steady-state part), in deg/s;
2. plotted $\omega$ against PWM % with a linear fit, slope $k$ in deg/s per 1 % PWM;
3. computed the coefficient $U/\omega$ in V·s/deg for every power level and averaged it.

Here we repeat exactly that procedure on our data, so the numbers can be put next to theirs.

In [ ]:
def compute_omega(positions, times):              # same as last year's plot.py
    n = len(positions)
    if n < 4:
        return 0.0
    start = int(n * 0.4)                            # skip the first 40 % (acceleration)
    dp = positions[-1] - positions[start]
    dt = (times[-1] - times[start])
    return dp / dt if dt != 0 else 0.0

ly = {}                                              # last-year style results per wheel
for wheel in ('L', 'R'):
    pct   = np.array([r['power'] for r in runs])
    U     = np.array([r['U'] for r in runs])
    omega = np.array([compute_omega(np.rad2deg(r[wheel]), r['time']) for r in runs])  # deg/s
    ok    = omega != 0                               # U/w is undefined where the wheel did not move
    ly[wheel] = {'pct': pct, 'U': U, 'omega': omega, 'coeff': U[ok] / omega[ok], 'coeff_pct': pct[ok]}

fig, ax = plt.subplots(2, 2, figsize=(14, 10), dpi=100)
for i, wheel in enumerate(('L', 'R')):
    d = ly[wheel]
    k, b = np.polyfit(d['pct'], d['omega'], 1)      # w = k * PWM% + b
    x = np.linspace(-100, 100, 200)
    ax[i, 0].scatter(d['pct'], d['omega'], color='#378ADD', s=40, zorder=3, label='data')
    ax[i, 0].plot(x, k * x + b, 'r--', label=f'linear fit: k = {k:.2f}')
    ax[i, 0].set_xlabel('PWM, %'); ax[i, 0].set_ylabel('w, deg/s')
    ax[i, 0].set_title(f'Wheel {wheel}: w vs PWM %'); ax[i, 0].grid(alpha=0.3); ax[i, 0].legend()

    avg = d['coeff'].mean()
    ax[i, 1].bar(d['coeff_pct'], d['coeff'], color='#1D9E75', width=3.5, zorder=3)
    ax[i, 1].axhline(avg, color='r', linestyle='--', label=f'mean = {avg:.5f}')
    ax[i, 1].set_xlabel('PWM, %'); ax[i, 1].set_ylabel('U/w, V*s/deg')
    ax[i, 1].set_title(f'Wheel {wheel}: coefficient U/w'); ax[i, 1].grid(alpha=0.3); ax[i, 1].legend()
    d['k'] = k
plt.tight_layout()

Same as last year, $U/\omega$ is not constant: it drops near zero power, where the motor speed is disproportionately high for the small applied voltage. The plain mean is therefore pulled down by the low-power runs. The table below gives both the plain mean (last year's number) and the mean over $|PWM| \geq 50\,\%$, together with the curve-fit $k_e$ from earlier in this notebook, converted to the same units ($1\ \text{V·s/rad} = \frac{\pi}{180}\ \text{V·s/deg}$).

In [ ]:
print('Last year (comments.md):')
print('  wheel   k, deg/s per %   mean U/w, V*s/deg')
for wid, k, c in ((1, 8.44, 0.02016), (2, 8.95, 0.01069), (3, 8.99, 0.01048), (4, 7.90, 0.01289), ('5 (no wheel)', 9.02, 0.01026)):
    print(f'  {str(wid):12s} {k:6.2f}         {c:.5f}')

print('\nThis year (trik_data_bat.txt):')
print('  wheel   k, deg/s per %   mean U/w   mean U/w |PWM|>=50%   curve-fit k_e |PWM|>=50%   (all V*s/deg)')
for wheel in ('L', 'R'):
    d = ly[wheel]
    hi_ly = np.abs(d['coeff_pct']) >= 50
    m = (res_wheel == wheel) & (np.abs(res_power) >= 50)
    k_e_fit = np.median(res_U[m] / res_w[m]) * np.pi / 180          # V*s/rad -> V*s/deg
    print(f'  {wheel:12s} {d["k"]:6.2f}         {d["coeff"].mean():.5f}    {d["coeff"][hi_ly].mean():.5f}'
          f'               {k_e_fit:.5f}')

Combined plot of $\omega(U)$ for both wheels, like last year's `omega_all_wheels.png`. The steady-state speeds from the slope method (lines) and from the exponential curve fit (crosses) should lie on top of each other.

In [ ]:
plt.figure(figsize=(12, 6), dpi=100)
for wheel, c in (('L', '#378ADD'), ('R', '#E24B4A')):
    d = ly[wheel]
    plt.plot(d['U'], d['omega'], 'o-', color=c, markersize=4, linewidth=1.2, label=f'wheel {wheel}, slope of last 60 %')
    m = res_wheel == wheel
    plt.plot(res_U[m], np.rad2deg(res_w[m]), 'x', color='k' if wheel == 'L' else 'gray', label=f'wheel {wheel}, curve fit w_ss')
plt.axhline(0, color='gray', linewidth=0.8, linestyle=':')
plt.axvline(0, color='gray', linewidth=0.8, linestyle=':')
plt.xlabel('U, V'); plt.ylabel('w, deg/s'); plt.title('All wheels: w vs U')
plt.grid(alpha=0.3); plt.legend()

## Numerical modelling
Now we need to become familiar with numerical modeling of motor rotation. In the following stages, we will need to work with a changing value of control voltage, which will be recalculated at each program cycle. First of all, we need to define a numerical model of the motor:
$$
\left\{
    \begin{array}\\
        \dot\omega=\frac{U}{T_m k_e}-\frac{\omega}{T_m}, \\
        \dot\theta=\omega.
    \end{array}
\right.
$$

In [ ]:
def motor(state, t):
    speed, angle = state                                             # motor state space
    state_dt = np.array([-speed / T_m + u / (T_m * k_e), speed])     # differential equation for the motor state
    return state_dt

The *odeint* function outputs the numerically integrated output of the model. Using the identified $T_m$ and $k_e$ and the measured voltage $U$ of the chosen run, *odeint* gives us the angular velocity and rotation angle of the motor. Let's check this on the graphs:

In [ ]:
init = [0, 0]                                                        # initial speed and angle
t_sim = np.linspace(0, time[-1], 200)                                # fine time grid for the simulation
y = odeint(motor, init, t_sim)                                       # numerical ode solution

fig, ax = plt.subplots(1, 2, figsize=(14, 5), dpi=100)
ax[0].plot(t_sim, y[:, 1], 'b-', label='odeint model')               # rotation angle plot
ax[0].plot(time, angle, 'r.', label='experiment')                    # experiment
ax[0].set_xlabel('t, s'); ax[0].set_ylabel('angle, rad'); ax[0].grid(); ax[0].legend()
ax[1].plot(t_sim, y[:, 0], 'b-', label='odeint model')               # motor speed plot
ax[1].plot(time[1:], np.diff(angle) / np.diff(time), 'r.', label='experiment (finite difference)')
ax[1].set_xlabel('t, s'); ax[1].set_ylabel('speed, rad/s'); ax[1].grid(); ax[1].legend()